# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. 5 (2, 3) почему так np.arange(6) создаёт числа от 0 до 5, после reshape(2, 3) получается 2 строки и 3 столбца. a[1, 2] — элемент во второй строке и третьем столбце, то есть 5
2. 1 8 rgmax() возвращает индекс максимального элемента. Максимальное значение 8 находится под индексом 1, а max() возвращает само значение 8
3. 99 a[::3] берёт каждый третий элемент, а срез является представлением исходного массива. Поэтому изменение b[0] = 99 изменяет и a[0].
4. 2 0.5 условие a > 3 выполняется только для 5 и 7, поэтому таких элементов 2. Среднее значение булевого массива равно 2/4 = 0.5.
5. (2, 3) 0.5 я правильно определил размер массива zeros как (2, 3), но ошибся при подсчёте второго элемента linspace. Я подумал, что значения идут с шагом 0.5, хотя на самом деле от 0 до 1 создаётся 5 равномерных значений с шагом 0.25.

In [7]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [8]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [9]:
# База
i = amount.argmax()
print(i, amount[i], cust[i], city[i])
# 2
print(amount[paid].sum(), amount[~paid].sum())
# 3
m = city == "Шымкент"
print(m.sum(), m.mean() * 100)

1001 9999990.0 TEST Алматы
101909360.0 7974000.0
804 14.71718835804503


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [10]:
# Стандарт
# 4
big = paid & (amount > 50000)
print(0.15 * amount[big].sum())
big2 = big & (cust != "TEST")
print(0.15 * amount[big2].sum())

# 5
for c in np.unique(city):
    mm = (city == c) & ~np.isnan(days)
    print(c, ((days[mm] >= 1) & (days[mm] <= 2)).mean() * 100)

# 6
ok = paid & (cust != "TEST")
print("Средний чек:", round(amount[ok].mean()))

5702422.5
1202427.0
Актобе 62.849872773536894
Алматы 66.50965614696185
Астана 65.4074074074074
Атырау 66.66666666666666
Караганда 71.01167315175097
Шымкент 30.69053708439898
Средний чек: 14672


## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [11]:
# Челлендж
# 7
import time
t = time.perf_counter()
s = 0
for k in range(len(amount)):
    if paid[k]:
        s += amount[k]
t_loop = time.perf_counter() - t

t = time.perf_counter()
s2 = amount[paid].sum()
t_mask = time.perf_counter() - t
print(t_loop / t_mask)

# 8
np.random.seed(42)
a = np.random.rand(1000) < 0.05
b = np.random.rand(1000) < 0.06
print(a.sum(), b.sum())

wins = 0
for _ in range(100):
    if (np.random.rand(1000) < 0.06).sum() > (np.random.rand(1000) < 0.05).sum():
        wins += 1
print(wins)

5.475073730829588
54 58
89
